# Plajah Motorsport: Apex — Rural Buildings Photoreal UV Painter & 2K Neural PBR Baker (Colab Pro+ GPU)

Paints all **6 Michigan rural hero building archetypes** (`rural_storefront_brick`, `rural_storefront_diner`, `rural_depot_hall`, `rural_barn_red`, `rural_farmhouse_clapboard`, `rural_church_civic`) locked pixel-for-pixel to their exact in-game UV blueprints, then bakes full **2048×2048 Neural PBR v4** maps (`_base`, `_nrm`, `_orm`, `_hgt`, `_rgh`) ready for `ingest_rural_omni_colab.mjs`!

### Two Built-In Painting Modes (Toggle in Cell 3)
1. **Mode A — `FLUX_OR_SDXL_CANNY_UV` (Default, Recommended):** Pixel-locked architectural painting via **`FLUX.1-dev + ControlNet-Canny`** (auto-enabled when `HF_TOKEN` is set in Colab Secrets) OR open-weights zero-token **`SDXL 1.0 + ControlNet-Canny-SDXL`** (runs cleanly with ZERO API tokens!). Guarantees every window, door, cornice, pilaster, shutter, and X-braced barn door lines up to the millimeter with the 3D geometry in `region.html`!
2. **Mode B — `ENABLE_HUNYUAN3D_360 = True` (Optional Full 360° Mesh Wrap):** Also runs **`tencent/Hunyuan3D-2` (`hunyuan3d-paint-v2-0-turbo`)** to paint the full 3D `.glb` mesh from all 6 angles using Mode A's photoreal facade as reference!

### Stage 2 — 2K Neural PBR Baker (Matches Downtown Ironport v4)
- **Real-ESRGAN x4plus (`spandrel`)** -> razor-sharp 2048×2048 brickwork, clapboard wood grain, painted signage & stone mortar
- **Marigold IID Appearance (`prs-eth/marigold-iid-appearance-v1-1`)** -> delit albedo + physical per-pixel roughness & metalness
- **Marigold Normals (`prs-eth/marigold-normals-v1-1`)** -> true OpenGL +Y-up 3D architectural normals RNM-blended with 2K micro-normals + Toksvig anti-sparkle roughness
- **Depth Anything V2 Large** -> real-time Parallax Occlusion Mapping height relief (`_hgt`)
- **Calibrated ORM (`_orm`)** -> `R = Cavity AO`, `G = Calibrated Roughness` (crisp glass window separation!), `B = Metalness`

**Before running:** Set **Runtime → Change runtime type → `A100 GPU` (recommended) or `L4 GPU`**, upload `tmp/apex-colab/rural_omni_inputs.zip` to `/content/`, and click **Runtime → Run all**.

In [ ]:
# Cell 1: Verify A100/L4 GPU and install wheel dependencies
import os, subprocess, sys, torch

assert torch.cuda.is_available(), 'No CUDA GPU attached! Set Runtime -> Change runtime type -> A100 GPU or L4 GPU.'
print(f'GPU Ready: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB VRAM)')

pkgs = [
    'diffusers>=0.32.0', 'transformers>=4.45.0', 'accelerate', 'safetensors',
    'timm', 'spandrel', 'opencv-python-headless', 'scipy', 'Pillow',
    'matplotlib', 'trimesh', 'tqdm'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)
import diffusers, transformers, spandrel, cv2
print('diffusers:', diffusers.__version__, '| transformers:', transformers.__version__, '| Ready!')

In [ ]:
# Cell 2: Locate & unpack rural_omni_inputs.zip (/content/ upload or Google Drive)
import glob, json, os, shutil, zipfile

USE_GOOGLE_DRIVE = False
INPUT_ZIP = None

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    cands = sorted(glob.glob('/content/drive/MyDrive/**/rural_omni_inputs*.zip', recursive=True), key=os.path.getmtime, reverse=True)
    if cands:
        INPUT_ZIP = cands[0]

if INPUT_ZIP is None:
    cands = sorted(glob.glob('/content/*rural_omni*inputs*.zip'), key=os.path.getmtime, reverse=True)
    if cands:
        INPUT_ZIP = cands[0]
    else:
        from google.colab import files
        print('Upload tmp/apex-colab/rural_omni_inputs.zip...')
        uploaded = files.upload()
        INPUT_ZIP = os.path.join('/content', next(iter(uploaded.keys())))

WORK_IN = '/content/rural_omni_in'
WORK_OUT = '/content/rural_omni_out'
if os.path.isdir(WORK_IN):
    shutil.rmtree(WORK_IN)
os.makedirs(WORK_IN, exist_ok=True)
os.makedirs(os.path.join(WORK_OUT, 'textures-v4', 'rural'), exist_ok=True)
os.makedirs(os.path.join(WORK_OUT, 'models-v4', 'rural'), exist_ok=True)

with zipfile.ZipFile(INPUT_ZIP, 'r') as zf:
    zf.extractall(WORK_IN)

MANIFEST = json.load(open(os.path.join(WORK_IN, 'manifest.json'), 'r', encoding='utf-8'))
print(f"Unpacked {len(MANIFEST['items'])} rural archetypes:")
for it in MANIFEST['items']:
    print('  -', it['id'], '->', it['label'])

In [ ]:
# Cell 3: Stage 1 — UV-Locked Photoreal Architectural Painting (FLUX.1-dev + ControlNet OR open-weights SDXL + ControlNet-Canny)
import gc, os, time
import numpy as np
import torch
from PIL import Image
from diffusers import ControlNetModel, StableDiffusionXLControlNetPipeline, AutoencoderKL

SEED = 42026
CONTROLNET_SCALE = 0.72          # Keeps every window, door, pilaster & X-brace locked to 3D UV coordinates
ENABLE_HUNYUAN3D_360 = False     # Set True to also wrap the full 3D .glb with Hunyuan3D-Paint 360°
PREFER_FLUX_IF_TOKEN = True      # Auto-uses FLUX.1-dev + ControlNet-Canny when HF_TOKEN secret is configured

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = None

PAINT_RAW_DIR = os.path.join(WORK_OUT, '_painted_raw')
os.makedirs(PAINT_RAW_DIR, exist_ok=True)

used_engine = 'sdxl-controlnet-canny'
pipe = None

if PREFER_FLUX_IF_TOKEN and HF_TOKEN:
    try:
        from diffusers import FluxControlNetModel, FluxControlNetPipeline
        print('HF_TOKEN detected -> loading FLUX.1-dev + InstantX ControlNet-Canny (bf16)...')
        cnet = FluxControlNetModel.from_pretrained(
            'InstantX/FLUX.1-dev-Controlnet-Canny', torch_dtype=torch.bfloat16, token=HF_TOKEN
        )
        pipe = FluxControlNetPipeline.from_pretrained(
            'black-forest-labs/FLUX.1-dev', controlnet=cnet, torch_dtype=torch.bfloat16, token=HF_TOKEN
        ).to('cuda')
        used_engine = 'flux1-dev-controlnet-canny'
    except Exception as e:
        print('FLUX load skipped (falling back to open-weights SDXL + ControlNet-Canny):', e)
        pipe = None

if pipe is None:
    print('Loading open-weights SDXL 1.0 + ControlNet-Canny-SDXL (fp16, zero token required)...')
    cnet = ControlNetModel.from_pretrained(
        'diffusers/controlnet-canny-sdxl-1.0', torch_dtype=torch.float16
    )
    vae = AutoencoderKL.from_pretrained('madebyollin/sdxl-vae-fp16-fix', torch_dtype=torch.float16)
    pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
        'stabilityai/stable-diffusion-xl-base-1.0', controlnet=cnet, vae=vae, torch_dtype=torch.float16
    ).to('cuda')
    used_engine = 'sdxl-controlnet-canny'

pipe.set_progress_bar_config(disable=True)
print(f'Active Neural Facade Painter: {used_engine}')

painted_paths = {}
for idx, item in enumerate(MANIFEST['items']):
    aid = item['id']
    out_png = os.path.join(PAINT_RAW_DIR, f'{aid}.png')
    t0 = time.time()
    canny_img = Image.open(os.path.join(WORK_IN, item['canny'])).convert('RGB')
    gen = torch.Generator(device='cuda').manual_seed(SEED + idx * 17)
    with torch.no_grad():
        if used_engine.startswith('flux'):
            img = pipe(
                prompt=item['prompt'],
                control_image=canny_img,
                controlnet_conditioning_scale=CONTROLNET_SCALE,
                num_inference_steps=28,
                guidance_scale=3.5,
                width=1024, height=1024,
                generator=gen,
            ).images[0]
        else:
            img = pipe(
                prompt=item['prompt'],
                negative_prompt=item['negativePrompt'],
                image=canny_img,
                controlnet_conditioning_scale=CONTROLNET_SCALE,
                num_inference_steps=34,
                guidance_scale=6.5,
                width=1024, height=1024,
                generator=gen,
            ).images[0]
    img.save(out_png)
    painted_paths[aid] = out_png
    print(f'  [{aid}] painted 1024px photoreal UV facade in {time.time() - t0:.1f}s')

del pipe
gc.collect(); torch.cuda.empty_cache()

# Optional Mode B: Hunyuan3D-2 Paint 360° wrap of the full 3D .glb meshes
if ENABLE_HUNYUAN3D_360:
    import glob, subprocess, sys, trimesh
    if not os.path.isdir('/content/Hunyuan3D-2'):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Tencent/Hunyuan3D-2.git', '/content/Hunyuan3D-2'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'xatlas', 'ninja', 'pybind11'], check=True)
    if '/content/Hunyuan3D-2' not in sys.path:
        sys.path.insert(0, '/content/Hunyuan3D-2')
    cr_dir = '/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', '-e', cr_dir], check=True)
    dr_dir = '/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer'
    if not glob.glob(os.path.join(dr_dir, 'mesh_processor*.so')):
        subprocess.run([sys.executable, 'setup.py', 'build_ext', '--inplace'], cwd=dr_dir, check=True)
    from hy3dgen.texgen import Hunyuan3DPaintPipeline
    hy_paint = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2', subfolder='hunyuan3d-paint-v2-0-turbo')
    for item in MANIFEST['items']:
        aid = item['id']
        mesh = trimesh.load(os.path.join(WORK_IN, item['mesh']), force='mesh')
        if isinstance(mesh, trimesh.Scene):
            mesh = mesh.dump(concatenate=True)
        ref_img = Image.open(painted_paths[aid]).convert('RGBA')
        painted_mesh = hy_paint(mesh, image=ref_img)
        out_glb = os.path.join(WORK_OUT, 'models-v4', 'rural', f'{aid}.glb')
        painted_mesh.export(out_glb)
        print(f'  [{aid}] Hunyuan3D 360° painted GLB -> {os.path.getsize(out_glb)/1e6:.2f} MB')
    del hy_paint
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 4: Stage 2 — 2K Neural PBR Engine (Real-ESRGAN x4plus + Marigold IID + Marigold Normals + Depth Anything V2 + Calibrated ORM)
import gc, os, time, urllib.request
import cv2
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
import diffusers
from transformers import pipeline as hf_pipeline
import spandrel

TEX_RURAL_OUT = os.path.join(WORK_OUT, 'textures-v4', 'rural')
MARIGOLD_STEPS = 4
MARIGOLD_ENSEMBLE = 2
DELIGHT_STRENGTH = 0.58

ESRGAN_PATH = '/content/RealESRGAN_x4plus.pth'
if not os.path.isfile(ESRGAN_PATH):
    urllib.request.urlretrieve(
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
        ESRGAN_PATH,
    )

def upscale_2k_tiled(rgb_u8: np.ndarray, esr_model, target_size: int = 2048) -> np.ndarray:
    h, w = rgb_u8.shape[:2]
    tile, ov, scale = 512, 32, 4
    acc = np.zeros((h * scale, w * scale, 3), dtype=np.float32)
    wt = np.zeros((h * scale, w * scale, 1), dtype=np.float32)
    with torch.no_grad():
        for y0 in range(0, h, tile - ov):
            y1 = min(h, y0 + tile); y0 = max(0, y1 - tile)
            for x0 in range(0, w, tile - ov):
                x1 = min(w, x0 + tile); x0 = max(0, x1 - tile)
                patch = rgb_u8[y0:y1, x0:x1]
                t_in = torch.from_numpy(patch).permute(2, 0, 1)[None].float().div(255.0).cuda().half()
                t_out = esr_model(t_in)[0].float().clamp(0, 1).permute(1, 2, 0).cpu().numpy() * 255.0
                ph, pw = t_out.shape[:2]
                w2 = np.maximum(0.05, np.hanning(ph + 2)[1:-1, None] * np.hanning(pw + 2)[None, 1:-1])[:, :, None].astype(np.float32)
                acc[y0*scale:y1*scale, x0*scale:x1*scale] += t_out * w2
                wt[y0*scale:y1*scale, x0*scale:x1*scale] += w2
    up4 = np.clip(acc / np.maximum(wt, 1e-5), 0, 255).astype(np.uint8)
    return cv2.resize(up4, (target_size, target_size), interpolation=cv2.INTER_AREA)

def rnm_blend(n_base: np.ndarray, n_detail: np.ndarray) -> np.ndarray:
    t = n_base.copy(); t[:, :, 2] += 1.0
    u = n_detail.copy(); u[:, :, 0] *= -1.0; u[:, :, 1] *= -1.0
    r = t * np.sum(t * u, axis=-1, keepdims=True) / np.maximum(t[:, :, 2:3], 1e-4) - u
    return r / np.maximum(np.linalg.norm(r, axis=-1, keepdims=True), 1e-6)

def toksvig_roughness(rgh01: np.ndarray, nrm_unit: np.ndarray, strength: float = 0.42) -> np.ndarray:
    n_avg = cv2.GaussianBlur(nrm_unit.astype(np.float32), (0, 0), 1.35)
    n_len = np.clip(np.linalg.norm(n_avg, axis=-1), 0.25, 1.0)
    var = np.clip((1.0 - n_len) / np.maximum(n_len, 0.25), 0.0, 0.65)
    return np.clip(np.sqrt(rgh01 * rgh01 + var * strength), 0.04, 0.98)

# Pass 1: Real-ESRGAN 2K + Depth Anything V2 Large
esr_model = spandrel.ModelLoader().load_from_file(ESRGAN_PATH).cuda().eval().half()
depth_pipe = hf_pipeline('depth-estimation', model='depth-anything/Depth-Anything-V2-Large-hf', device=0, torch_dtype=torch.float16)
sr_cache, depth_cache = {}, {}
for item in MANIFEST['items']:
    aid = item['id']
    rgb = np.array(Image.open(painted_paths[aid]).convert('RGB'))
    rgb2k = upscale_2k_tiled(rgb, esr_model, 2048)
    sr_cache[aid] = rgb2k
    im_d = Image.fromarray(cv2.resize(rgb2k, (1024, 1024), interpolation=cv2.INTER_AREA))
    with torch.no_grad():
        d_t = depth_pipe(im_d)['predicted_depth']
    d_raw = d_t.detach().float().cpu().numpy().astype(np.float32) if hasattr(d_t, 'detach') else np.asarray(d_t, dtype=np.float32)
    d_raw = cv2.resize(d_raw, (1024, 1024), interpolation=cv2.INTER_CUBIC)
    p2, p98 = float(np.percentile(d_raw, 2)), float(np.percentile(d_raw, 98))
    depth_cache[aid] = np.clip((d_raw - p2) / max(1e-5, p98 - p2), 0.0, 1.0)
    print(f'  [2K SR + Depth] {aid} OK')
del esr_model, depth_pipe; gc.collect(); torch.cuda.empty_cache()

# Pass 2: Marigold IID Appearance
iid_pipe = diffusers.MarigoldIntrinsicsPipeline.from_pretrained(
    'prs-eth/marigold-iid-appearance-v1-1', variant='fp16', torch_dtype=torch.float16
).to('cuda')
iid_pipe.set_progress_bar_config(disable=True)
iid_cache = {}
for aid, rgb2k in sr_cache.items():
    im_in = Image.fromarray(cv2.resize(rgb2k, (768, 768), interpolation=cv2.INTER_AREA))
    with torch.no_grad():
        out = iid_pipe(im_in, num_inference_steps=MARIGOLD_STEPS, ensemble_size=MARIGOLD_ENSEMBLE)
    vis = iid_pipe.image_processor.visualize_intrinsics(out.prediction, iid_pipe.target_properties)[0]
    iid_cache[aid] = (
        np.asarray(vis['albedo'].convert('RGB'), dtype=np.float32) / 255.0,
        np.asarray(vis['roughness'].convert('L'), dtype=np.float32) / 255.0,
        np.asarray(vis['metallicity'].convert('L'), dtype=np.float32) / 255.0,
    )
    print(f'  [Marigold IID] {aid} OK')
del iid_pipe; gc.collect(); torch.cuda.empty_cache()

# Pass 3: Marigold Normals + Calibrated PBR Export
nrm_pipe = diffusers.MarigoldNormalsPipeline.from_pretrained(
    'prs-eth/marigold-normals-v1-1', variant='fp16', torch_dtype=torch.float16
).to('cuda')
nrm_pipe.set_progress_bar_config(disable=True)
qa_rows = []
for item in MANIFEST['items']:
    aid = item['id']
    rgb2k = sr_cache[aid]
    im_in = Image.fromarray(cv2.resize(rgb2k, (768, 768), interpolation=cv2.INTER_AREA))
    with torch.no_grad():
        nout = nrm_pipe(im_in, num_inference_steps=MARIGOLD_STEPS, ensemble_size=MARIGOLD_ENSEMBLE)
    n_pred = np.squeeze(nout.prediction).astype(np.float32)
    if n_pred.shape[0] == 3 and n_pred.ndim == 3:
        n_pred = np.transpose(n_pred, (1, 2, 0))
    n_base = cv2.resize(n_pred, (2048, 2048), interpolation=cv2.INTER_CUBIC)
    n_base[:, :, 2] = np.abs(n_base[:, :, 2]) + 0.12
    n_base /= np.maximum(np.linalg.norm(n_base, axis=-1, keepdims=True), 1e-6)

    # Frequency-separated delighting (keeps crisp 2K mortar & plank detail)
    alb_lo, rgh_lo, mtl_lo = iid_cache[aid]
    rgb_f = rgb2k.astype(np.float32) / 255.0
    alb_up = cv2.resize(alb_lo, (2048, 2048), interpolation=cv2.INTER_CUBIC)
    rgb_blur = cv2.GaussianBlur(rgb_f, (0, 0), 18.0)
    alb_blur = cv2.GaussianBlur(alb_up, (0, 0), 18.0)
    delit = np.clip(rgb_f + (alb_blur - rgb_blur) * DELIGHT_STRENGTH, 0.0, 1.0)

    # 2K tactile micro-normal RNM blend
    lum = cv2.cvtColor((delit * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    gx = cv2.Sobel(lum, cv2.CV_32F, 1, 0, ksize=3) * 1.85
    gy = cv2.Sobel(lum, cv2.CV_32F, 0, 1, ksize=3) * 1.85
    n_mic = np.stack([-gx, gy, np.ones_like(gx)], axis=-1)
    n_mic /= np.maximum(np.linalg.norm(n_mic, axis=-1, keepdims=True), 1e-6)
    n_final = rnm_blend(n_base, n_mic)
    nrm_u8 = np.clip((n_final * 0.5 + 0.5) * 255.0, 0, 255).astype(np.uint8)

    # Height + Cavity AO + Window Glass Separation into calibrated ORM
    hgt_2k = cv2.resize(depth_cache[aid], (2048, 2048), interpolation=cv2.INTER_CUBIC)
    hgt_lo = cv2.GaussianBlur(hgt_2k, (0, 0), 14.0)
    ao = np.clip(0.86 + (hgt_2k - hgt_lo) * 0.95, 0.38, 1.0)
    # Blueprint blue-glass regions get crisp architectural glass roughness (~0.10) automatically!
    bp_rgb = cv2.resize(np.array(Image.open(os.path.join(WORK_IN, item['blueprint'])).convert('RGB')), (2048, 2048), interpolation=cv2.INTER_NEAREST)
    is_win = ((bp_rgb[:, :, 2].astype(np.int16) - bp_rgb[:, :, 0].astype(np.int16)) > 22) & (bp_rgb[:, :, 0] < 72)
    is_win_f = cv2.GaussianBlur(is_win.astype(np.float32), (0, 0), 2.5)

    rgh_up = cv2.resize(rgh_lo, (2048, 2048), interpolation=cv2.INTER_CUBIC)
    rgh_cal = np.clip(rgh_up * 0.55 + item['roughnessBase'] * 0.45, 0.35, 0.96)
    rgh_cal = toksvig_roughness(rgh_cal, n_final, 0.40)
    rgh_cal = rgh_cal * (1.0 - is_win_f) + 0.09 * is_win_f
    mtl_cal = np.clip(cv2.resize(mtl_lo, (2048, 2048), interpolation=cv2.INTER_CUBIC) * 0.35 + item['metalnessBase'] * 0.65, 0.0, 0.45)
    mtl_cal = np.maximum(mtl_cal, is_win_f * 0.16)

    orm_u8 = np.stack([(ao * 255).astype(np.uint8), (rgh_cal * 255).astype(np.uint8), (mtl_cal * 255).astype(np.uint8)], axis=-1)
    rgh_u8 = (rgh_cal * 255).astype(np.uint8)
    hgt_u8 = (hgt_2k * 255).astype(np.uint8)
    base_u8 = (delit * 255).astype(np.uint8)

    Image.fromarray(base_u8).save(os.path.join(TEX_RURAL_OUT, f'{aid}_base.jpg'), quality=94, subsampling=0)
    Image.fromarray(nrm_u8).save(os.path.join(TEX_RURAL_OUT, f'{aid}_nrm.jpg'), quality=94, subsampling=0)
    Image.fromarray(orm_u8).save(os.path.join(TEX_RURAL_OUT, f'{aid}_orm.jpg'), quality=94, subsampling=0)
    Image.fromarray(rgh_u8).save(os.path.join(TEX_RURAL_OUT, f'{aid}_rgh.jpg'), quality=92)
    Image.fromarray(hgt_u8).save(os.path.join(TEX_RURAL_OUT, f'{aid}_hgt.jpg'), quality=92)
    qa_rows.append((aid, base_u8, nrm_u8, orm_u8, hgt_u8))
    print(f'  [Exported 2K PBR] {aid} -> _base, _nrm, _orm, _rgh, _hgt')

del nrm_pipe; gc.collect(); torch.cuda.empty_cache()

# Save visual QA contact sheet
fig, axes = plt.subplots(len(qa_rows), 4, figsize=(16, 4 * len(qa_rows)))
for r, (aid, b, n, o, h) in enumerate(qa_rows):
    for c, (img, title) in enumerate([(b, f'{aid} Albedo'), (n, 'Marigold Normal'), (o, 'Calibrated ORM'), (h, 'Depth Height')]):
        ax = axes[r, c] if len(qa_rows) > 1 else axes[c]
        ax.imshow(img, cmap='gray' if c == 3 else None)
        ax.set_title(title, fontsize=10)
        ax.axis('off')
plt.tight_layout()
qa_png = os.path.join(WORK_OUT, 'rural_pbr_qa_preview.png')
plt.savefig(qa_png, dpi=130)
plt.show()

In [ ]:
# Cell 5: Bundle rural_omni_v4_outputs.zip & trigger download
import os, zipfile
from google.colab import files

OUT_ZIP = '/content/rural_omni_v4_outputs.zip'
with zipfile.ZipFile(OUT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for root, _, fnames in os.walk(WORK_OUT):
        if '_painted_raw' in root:
            continue
        for fn in sorted(fnames):
            full = os.path.join(root, fn)
            zf.write(full, os.path.relpath(full, WORK_OUT))

print(f'READY: {OUT_ZIP} ({os.path.getsize(OUT_ZIP) / 1e6:.2f} MB)')
print('On your PC, run:')
print('  node games/apex/tools/colab/ingest_rural_omni_colab.mjs path/to/rural_omni_v4_outputs.zip')
files.download(OUT_ZIP)